# Problem 3 — Many-Time Pad Encryption

Encrypt ten predefined ASCII messages using one shared key. Save plaintexts, key, and ciphertexts as hexadecimal strings in one JSON file.

The messages below are editable examples. Replace them with your own chosen messages if desired. This simulation uses known plaintext and a known key for learning; Problem 4 uses the separately supplied dataset.

Use Python 3.9 or newer. This notebook also runs as a standalone download. It creates a `data/` output folder in the kernel's working directory, or uses `Many_Time_Pad/data/` when started from the repository root. The first code cell prints the output folder.


In [ ]:
from pathlib import Path
import json
import secrets

MESSAGES = [
    "Meet me near the bridge.",
    "The river is quiet today.",
    "Can you see the morning light?",
    "We should compare our notes.",
    "A secret key must stay secret.",
    "Please bring the notebook.",
    "The train will arrive soon.",
    "Keep the messages in order.",
    "What changes when we reuse a key?",
    "Now we can observe the pattern.",
]

# Use the repository's Many_Time_Pad folder when available.
# A standalone copy saves relative to the kernel's working directory.
workspace = Path("Many_Time_Pad") if Path("Many_Time_Pad").is_dir() else Path(".")
(workspace / "data").mkdir(parents=True, exist_ok=True)
print("Simulation output folder:", (workspace / "data").resolve())

if len(MESSAGES) != 10 or any(not message for message in MESSAGES):
    raise ValueError("Provide exactly ten nonempty messages.")
plaintexts = [message.encode("ascii") for message in MESSAGES]


## One key reused across all ten messages

Generate enough bytes for the longest message. Each message starts with key byte zero and uses the matching prefix. Re-running with a new key changes the ciphertexts, but the XOR of two ciphertexts remains the XOR of their plaintexts.


In [ ]:
def encrypt_with_shared_key(plaintexts, key):
    if not plaintexts or len(key) < max(map(len, plaintexts)):
        raise ValueError("The shared key must cover the longest message.")
    return [bytes(byte ^ key[offset] for offset, byte in enumerate(plaintext))
            for plaintext in plaintexts]


# Generate once, then reuse this same key for all ten messages.
key = secrets.token_bytes(max(map(len, plaintexts)))
ciphertexts = encrypt_with_shared_key(plaintexts, key)

for index, (plaintext, ciphertext) in enumerate(zip(plaintexts, ciphertexts), 1):
    recovered = bytes(byte ^ key[offset] for offset, byte in enumerate(ciphertext))
    assert recovered == plaintext, f"Round-trip failure in message {index}."
    print(f"M{index:02d}: {plaintext.decode('ascii')}")
    print("Plaintext hex:", plaintext.hex())
    print("Ciphertext hex:", ciphertext.hex())
print("Shared key hex:", key.hex())


## Save the required hexadecimal artifact

This cell writes only `data/simulation.json`; it does not modify the official Problem 4 inputs. A rerun replaces the simulation artifact. All plaintext, key, and ciphertext values are stored in hexadecimal form; labels and lengths are metadata.


In [ ]:
artifact = {
    "exercise": "Problem 3 simulation",
    "encoding": "ASCII plaintext; hex byte values",
    "key_hex": key.hex(),
    "messages": [
        {
            "label": f"M{index:02d}",
            "byte_length": len(plaintext),
            "plaintext_hex": plaintext.hex(),
            "ciphertext_hex": ciphertext.hex(),
        }
        for index, (plaintext, ciphertext) in enumerate(zip(plaintexts, ciphertexts), 1)
    ],
}
OUTPUT_FILE = workspace / "data" / "simulation.json"
OUTPUT_FILE.write_text(json.dumps(artifact, indent=2) + "\n", encoding="ascii")
print(f"Saved ten messages, one shared key, and ten ciphertexts to {OUTPUT_FILE}")


## Observe key cancellation

Change LEFT_MESSAGE and RIGHT_MESSAGE to compare a different pair. Test modified plaintexts or a different key and record your observations for the written report.


In [ ]:
LEFT_MESSAGE = 1
RIGHT_MESSAGE = 2
if not (1 <= LEFT_MESSAGE <= 10 and 1 <= RIGHT_MESSAGE <= 10):
    raise ValueError("Choose message numbers from 1 through 10.")
if LEFT_MESSAGE == RIGHT_MESSAGE:
    raise ValueError("Select two different messages.")

def xor_overlap(left, right):
    return bytes(a ^ b for a, b in zip(left, right))

left = LEFT_MESSAGE - 1
right = RIGHT_MESSAGE - 1
ciphertext_xor = xor_overlap(ciphertexts[left], ciphertexts[right])
plaintext_xor = xor_overlap(plaintexts[left], plaintexts[right])
assert ciphertext_xor == plaintext_xor

print("Ciphertext XOR:", ciphertext_xor.hex())
print("Plaintext XOR: ", plaintext_xor.hex())
print("They match: the reused key cancels over the overlapping positions.")

alternative_key = secrets.token_bytes(len(key))
alternative_ciphertexts = encrypt_with_shared_key(plaintexts, alternative_key)
assert xor_overlap(alternative_ciphertexts[left], alternative_ciphertexts[right]) == plaintext_xor
print("The same relation also holds after changing the shared key.")


## Observations to complete

- What remains invariant when the key changes?
- What happens at a position where two messages have the same character?
- How does a space XORed with a letter affect its ASCII case?
- How do unequal message lengths affect the available comparisons?
- Which observations suggest a strategy for Problem 4?

Record actual experiments and outcomes in the report. This notebook implements the simulation; your interpretation and team review are still required.
